# Negative Binomial Regression - BMWP Prediction

The BMWP/Col index is an over-dispersed count, so a negative binomial GLM is used (rather than Poisson). Predictors are screened by AIC; the selected model is validated with leave-one-out cross-validation, mapping predicted values onto the BMWP quality classes.

## Data Loading

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from itertools import combinations
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

file_path = "../../data/Database - BMWP.xlsx"
df = pd.read_excel(file_path)
df.columns = df.columns.str.strip()
df

## Predictor Selection

Seven candidate predictors are considered for the BMWP response.

In [ ]:
predictors = ['COT', 'DBO5', 'Dureza', 'Magnesio', 'Turbiedad', 'OD', 'Caudal']
response = ['BMWP']
X = df[predictors]
y = df[response]

In [ ]:
# Overdispersion check: variance >> mean justifies negative binomial over Poisson
mean_bmwp = df['BMWP'].mean()
var_bmwp = df['BMWP'].var()
print("BMWP mean:", mean_bmwp)
print("BMWP variance:", var_bmwp)
print(f"Variance / mean ratio: {var_bmwp / mean_bmwp:.2f}")

### Saturated model

In [ ]:
X_const = sm.add_constant(X)
saturated = sm.GLM(y, X_const, family=sm.families.NegativeBinomial()).fit()
print(saturated.summary())

### AIC-based subset selection

All predictor subsets are fitted and ranked by AIC. The best model for BMWP uses `Dureza` (total hardness).

In [ ]:
def select_model(dataframe, predictors, response):
    best_model = None
    best_aic = np.inf
    best_combination = None
    total_combinations = 0

    for L in range(1, len(predictors) + 1):
        for subset in combinations(predictors, L):
            total_combinations += 1
            X_subset = sm.add_constant(dataframe[list(subset)])
            result = sm.GLM(dataframe[response], X_subset,
                            family=sm.families.NegativeBinomial()).fit()
            print(f'Trying model with predictors: {subset}, AIC: {result.aic}')
            if result.aic < best_aic:
                best_aic = result.aic
                best_model = result
                best_combination = subset

    print(f'Total combinations tried: {total_combinations}')
    print(f'Best model has predictors: {best_combination} with AIC: {best_aic}')
    return best_model

best_model = select_model(df, predictors, response)
print(best_model.summary())

## Model Definition & Evaluation

Leave-one-out cross-validation of the negative binomial model with `Dureza`. Continuous predictions are mapped to the five BMWP/Col quality classes, and calibration/validation confusion matrices and reports are produced.

In [ ]:
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

bmwp_classes = {
    'Buena': (101, 120),
    'Aceptable': (61, 100),
    'Dudosa': (36, 60),
    'Crítica': (16, 35),
    'Muy Crítica': (0, 15),
}

def classify_bmwp(value):
    for label, (lower, upper) in bmwp_classes.items():
        if lower <= value <= upper:
            return label
    return None

X = df[['Dureza']]
y = df['BMWP']
loo = LeaveOneOut()

y_true_val, y_pred_val = [], []
y_true_cal, y_pred_cal = [], []

for train_index, val_index in loo.split(X):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    X_train_const = sm.add_constant(X_train, has_constant='add')
    X_val_const = sm.add_constant(X_val, has_constant='add')

    results = sm.GLM(y_train, X_train_const, family=sm.families.NegativeBinomial()).fit()

    y_pred_cal += results.predict(X_train_const).tolist()
    y_true_cal += y_train.tolist()
    y_pred_val.append(results.predict(X_val_const).iloc[0])
    y_true_val.append(y_val.iloc[0])

# Map continuous predictions to BMWP classes
y_pred_val_class = [classify_bmwp(v) for v in y_pred_val]
y_true_val_class = [classify_bmwp(v) for v in y_true_val]
y_pred_cal_class = [classify_bmwp(v) for v in y_pred_cal]
y_true_cal_class = [classify_bmwp(v) for v in y_true_cal]

# Drop any None (out-of-range) entries
def drop_none(true_list, pred_list):
    idx = [i for i, (t, p) in enumerate(zip(true_list, pred_list)) if t is not None and p is not None]
    return [true_list[i] for i in idx], [pred_list[i] for i in idx]

y_true_cal_class, y_pred_cal_class = drop_none(y_true_cal_class, y_pred_cal_class)
y_true_val_class, y_pred_val_class = drop_none(y_true_val_class, y_pred_val_class)

labels = list(bmwp_classes.keys())
cal_matrix = confusion_matrix(y_true_cal_class, y_pred_cal_class, labels=labels)
val_matrix = confusion_matrix(y_true_val_class, y_pred_val_class, labels=labels)

def plot_confusion_matrix(cm, labels, title):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', annot_kws={"size": 16},
                xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Real')
    plt.title(title)
    plt.show()

plot_confusion_matrix(cal_matrix, labels, 'Confusion Matrix (Calibration)')
plot_confusion_matrix(val_matrix, labels, 'Confusion Matrix (Validation)')

print("Classification report - calibration:\n")
print(classification_report(y_true_cal_class, y_pred_cal_class, labels=labels, target_names=labels))
print("Classification report - validation:\n")
print(classification_report(y_true_val_class, y_pred_val_class, labels=labels, target_names=labels))

## Visualisation

Fitted negative binomial curve of BMWP as a function of total hardness.

In [ ]:
# Refit on the full sample for the fitted-curve plot
X_full = sm.add_constant(df[['Dureza']], has_constant='add')
results = sm.GLM(df['BMWP'], X_full, family=sm.families.NegativeBinomial()).fit()

intercept = results.params['const']
coef_hardness = results.params['Dureza']

hardness_continuous = np.linspace(df['Dureza'].min(), df['Dureza'].max(), 100)
X_pred = sm.add_constant(pd.DataFrame({'Dureza': hardness_continuous}), has_constant='add')
bmwp_predicted = results.predict(X_pred)

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(df['Dureza'], df['BMWP'], label='Observed values', color='blue', s=50)
ax.plot(hardness_continuous, bmwp_predicted, color='red', linewidth=2,
        label='Fitted model (Negative Binomial Regression)')
ax.set_xlabel('Total Hardness (mg CaCO₃/L)', fontsize=13)
ax.set_ylabel('BMWP', fontsize=13)
ax.set_title('Total Hardness vs BMWP (Negative Binomial Regression)', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True)
equation_text = rf"$\mu = e^{{{intercept:.4f} + {coef_hardness:.4f} \cdot \mathrm{{Hardness}}}}$"
ax.text(0.55, 0.9, equation_text, transform=ax.transAxes, fontsize=15, color='black',
        bbox=dict(facecolor='white', edgecolor='black', alpha=0.5))
fig.savefig("../../figures/negative_binomial_BMWP.png", dpi=300, bbox_inches='tight')
plt.show()